Updated : 2026-06-23

# Objective
Combine data into single data frame
 - Weekly
 - Monthly
 - Spend (Excluded from analysis - Yearly aggregated data)

In [9]:
# Import libraries
import pandas as pd

In [10]:
workbook_path = "data\\IND DB (FY20 - FY23).xlsx"
df_weekly = pd.read_excel(workbook_path,sheet_name="weekly") # "week" represents date.
df_monthly = pd.read_excel(workbook_path,sheet_name="monthly") # "month" represents date.

In [11]:
# --- checks + combine (weekly, monthly, spend) ---
import pandas as pd
import numpy as np
from pathlib import Path

# 1) Basic file/sheet checks
wb = Path(workbook_path)
if not wb.exists():
    raise FileNotFoundError(f"Workbook not found: {wb.resolve()}")

xls = pd.ExcelFile(workbook_path)
required_sheets = {"weekly", "monthly", "spend"}
missing_sheets = required_sheets - set(xls.sheet_names)
if missing_sheets:
    raise ValueError(f"Missing sheets in workbook: {missing_sheets}")

# 2) Standardize column names
def _clean_cols(df):
    out = df.copy()
    out.columns = [str(c).strip().lower() for c in out.columns]
    return out

df_weekly = _clean_cols(df_weekly)
df_monthly = _clean_cols(df_monthly)

if df_weekly.empty or df_monthly.empty:
    raise ValueError("One or more input dataframes are empty.")

# 3) Detect/parse date columns
def _pick_date_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of {candidates} found in columns: {list(df.columns)}")

wk_date_col = _pick_date_col(df_weekly, ["week", "date", "week_start"])
mo_date_col = _pick_date_col(df_monthly, ["month", "date"])

for dfx, c, nm in [(df_weekly, wk_date_col, "weekly"),
                   (df_monthly, mo_date_col, "monthly")]:
    dfx[c] = pd.to_datetime(dfx[c], errors="coerce")
    if dfx[c].isna().any():
        bad = dfx[c].isna().sum()
        raise ValueError(f"{nm}: {bad} invalid date values in '{c}'")

# 4) Create common monthly key
df_weekly["month"] = df_weekly[wk_date_col].dt.to_period("M").dt.to_timestamp()
df_monthly["month"] = df_monthly[mo_date_col].dt.to_period("M").dt.to_timestamp()

# 5) Infer join keys (non-date common columns)
date_like = {wk_date_col, mo_date_col,"month"}
common_cols = set(df_weekly.columns) & set(df_monthly.columns)
id_cols = sorted([c for c in common_cols if c not in date_like])

# keep mostly categorical/id-like columns for joins
id_cols = [c for c in id_cols if not pd.api.types.is_numeric_dtype(df_monthly[c])]
if not id_cols:
    print("Warning: no shared ID columns found. Merging on 'month' only.")

merge_keys = id_cols + ["month"]

# 6) Aggregate to unique grain before merge
def _agg_numeric(df, keys):
    num_cols = [c for c in df.columns if c not in keys and pd.api.types.is_numeric_dtype(df[c])]
    if not num_cols:
        return df[keys].drop_duplicates()
    out = df.groupby(keys, dropna=False, as_index=False)[num_cols].sum(min_count=1)
    return out

weekly_m  = _agg_numeric(df_weekly, merge_keys)
monthly_m = _agg_numeric(df_monthly, merge_keys)


# 7) Merge with validation checks
# (m:m used because monthly duplicates may exist before aggregation across non-numeric fields)
df_combined = (
    weekly_m
    .merge(monthly_m, on=merge_keys, how="outer", validate="m:m", suffixes=("_weekly", "_monthly"))
    .sort_values(merge_keys)
    .reset_index(drop=True)
)

# 8) Final QA
dup_count = df_combined.duplicated(merge_keys).sum()
if dup_count:
    print(f"Warning: {dup_count} duplicate rows remain at merge grain: {merge_keys}")

print("Combine complete.")
print(f"Merge keys: {merge_keys}")
print(f"Shape: {df_combined.shape}")
display(df_combined.head())

Combine complete.
Merge keys: ['month']
Shape: (44, 104)


,month,dig_aud,video_io,video_prog,display_io,display_prog,social_display,social_video,search_off_weekly,search_def_weekly,...,web_visits_ut,cpi,petrol_price,unemp,oao_ut,oao_magnite,cftp,sales_div_tiv,pct_discount,fy
0,2020-08-01,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-09-01,0.0,0.000000,36.406696,79.648201,0.000000,0.000000,289.167147,0,5831,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-10-01,0.0,1.968820,11.315008,122.344295,12.974211,0.717193,119.713601,67146,138566,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-11-01,0.0,2.629429,31.187527,298.387278,107.356432,199.597142,174.602798,68074,589115,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-12-01,0.0,6.914424,330.766554,824.717036,301.854822,150.002479,41.835562,156672,808524,...,4256183.0,130.8891,86.272742,9.1,0.0,0.0,766100.810793,0.009877,0.002365,2020.0


In [12]:
df_combined.shape

(44, 104)

In [17]:
df_combined.describe(include="all").T

,count,mean,min,25%,50%,75%,max,std
month,44,2022-05-17 03:49:05.454545,2020-08-01 00:00:00,2021-06-23 12:00:00,2022-05-16 12:00:00,2023-04-08 12:00:00,2024-03-01 00:00:00,NaN
dig_aud,44.0,0.738761,0.0,0.0,0.0,0.0,27.211099,4.160894
video_io,44.0,3.15066,0.0,0.0,0.0,2.133972,41.248258,7.571802
video_prog,44.0,33.858034,0.0,0.0,10.412518,41.730668,330.766554,59.356579
display_io,44.0,492.417269,0.0,292.583836,459.299428,680.36283,1294.5821,291.579849
...,...,...,...,...,...,...,...,...
oao_magnite,40.0,28.519371,0.0,25.606676,31.466667,34.002907,40.47,8.950692
cftp,40.0,792124.581203,685417.966849,731709.823398,806987.972685,845404.903961,865688.326154,55528.769788
sales_div_tiv,40.0,0.033034,0.009877,0.023939,0.031413,0.040542,0.060123,0.011069
pct_discount,40.0,0.018223,0.0,0.001765,0.004679,0.034909,0.080137,0.021743


In [ ]:
# df_combined.to_csv("data\\combined_data.csv", index=False)

In [8]:
print(os.path.join(BASE_DIR, "outputs", "result.json"))

NameError: name 'os' is not defined